# 03. 事件与进程机制

这一章深入解释 SimPy 的事件和进程机制。理解这一层后，阅读 `simpy.core` 与 `simpy.events` 的源码会容易很多。

## 事件是 SimPy 的最小调度单位

在 SimPy 中，所有等待都表现为事件：
```python
yield env.timeout(5)
yield resource.request()
yield store.get()
yield container.put(10)
yield another_process
```
这些对象虽然语义不同，但在调度层都遵循同一规则：

1. 进程 `yield` 一个事件。
2. 如果事件还没被触发，进程挂起。
3. 事件触发后，环境恢复等待它的进程。
4. 进程从 `yield` 处继续执行，并收到事件的值或异常。

## 事件生命周期

一个事件通常经历三个阶段：

| 阶段 | 说明 |
| --- | --- |
| 创建 | 事件对象被创建，尚未触发 |
| 触发 | 调用 `succeed()`、`fail()`，或由资源、超时、进程自动触发 |
| 处理 | 环境从事件队列取出事件，执行回调并恢复等待者 |

示例：

In [3]:
import simpy


def waiter(env, event):
    print(f"{env.now}: 等待事件")
    value = yield event
    print(f"{env.now}: 事件值 = {value}")


def trigger(env, event):
    yield env.timeout(3)
    event.succeed("ok")


env = simpy.Environment()
event = env.event()
env.process(waiter(env, event))
env.process(trigger(env, event))
env.run()

0: 等待事件
3: 事件值 = ok


## 成功事件和失败事件

事件可以成功，也可以失败。


In [4]:
import simpy


def consumer(env, event):
    try:
        value = yield event
        print("成功:", value)
    except RuntimeError as exc:
        print("失败:", exc)


env = simpy.Environment()
event = env.event()
env.process(consumer(env, event))
event.fail(RuntimeError("broken"))
env.run()

失败: broken


失败事件会把异常抛回等待它的进程。如果异常没有被处理，仿真会停止并抛出异常。


## Timeout

`env.timeout(delay, value=None)` 创建一个在 `delay` 时间后触发的事件。

```python
yield env.timeout(5)
value = yield env.timeout(2, value="done")
```

注意：

- `delay` 不能为负。
- `timeout` 创建后通常会立即被调度到未来事件队列。
- `value` 会作为 `yield` 表达式的返回值。

## Process 本身也是事件

`env.process(generator)` 返回一个 `Process` 对象。这个对象可以被 `yield`。


In [5]:
import simpy


def child(env):
    yield env.timeout(2)
    return 123


def parent(env):
    result = yield env.process(child(env))
    print(f"{env.now}: 子进程返回 {result}")


env = simpy.Environment()
env.process(parent(env))
env.run()

2: 子进程返回 123


当子进程结束时：

- 如果正常 `return`，进程事件成功，值是返回值。
- 如果抛出异常，进程事件失败。

## 进程的启动

`env.process()` 注册进程后，SimPy 会安排一个初始化事件，使进程在仿真开始时运行到第一个 `yield`。

这意味着下面两件事不同：
```python
gen = my_process(env)        # 只是创建 Python 生成器
proc = env.process(gen)      # 注册为 SimPy 进程
```
如果只创建生成器而不交给 `env.process()`，它不会被 SimPy 调度。


## 回调

事件内部有回调列表。事件被处理时，回调会被执行。进程等待事件，本质上也是给事件注册回调。

用户也可以手动追加回调：

In [6]:
import simpy


def callback(event):
    print("事件处理，值:", event.value)


env = simpy.Environment()
event = env.timeout(1, value="ready")
event.callbacks.append(callback)
env.run()

事件处理，值: ready


通常业务代码不需要直接操作回调。它更适合：

- 做底层监控。
- 给资源操作打点。
- 阅读源码时理解调度流程。

## 条件事件

条件事件用于组合多个事件。

### AnyOf：任意一个完成

```python
result = yield event_a | event_b
```

等价于等待 `event_a` 或 `event_b` 任意一个触发。返回值是一个字典式结果，里面包含已经完成的事件和对应值。


In [7]:
import simpy


def demo(env):
    slow = env.timeout(5, value="slow")
    fast = env.timeout(2, value="fast")

    result = yield slow | fast
    print(env.now, list(result.values()))


env = simpy.Environment()
env.process(demo(env))
env.run()

2 ['fast']


输出时间是 2，因为 `fast` 先完成。

### AllOf：全部完成

```python
result = yield event_a & event_b
```

只有所有事件都触发后，条件事件才触发。


In [8]:
import simpy


def demo(env):
    a = env.timeout(2, value="A")
    b = env.timeout(5, value="B")

    result = yield a & b
    print(env.now, list(result.values()))


env = simpy.Environment()
env.process(demo(env))
env.run()

5 ['A', 'B']


## 条件事件的典型用途

### 服务完成或超时

```python
with resource.request() as req:
    patience = env.timeout(10)
    result = yield req | patience

    if req in result:
        yield env.timeout(3)
    else:
        print("等待超时，离开队列")
```

注意：如果请求已经排队但顾客超时离开，应确保请求被取消。使用 `with` 管理请求通常能处理退出清理。


### 多个前置条件

```python
yield data_ready & model_ready & gpu_ready
```

适合表达“数据、模型和资源全部准备好后再执行”。


## 中断

进程可以中断另一个正在等待事件的进程。

In [10]:
import simpy


def worker(env):
    try:
        while True:
            print(f"{env.now}: 工作中")
            yield env.timeout(1)
    except simpy.Interrupt as interrupt:
        print(f"{env.now}: 被中断，原因 {interrupt.cause}")


def controller(env, proc):
    yield env.timeout(3)
    proc.interrupt("shutdown")


env = simpy.Environment()
proc = env.process(worker(env))
env.process(controller(env, proc))
env.run()

0: 工作中
1: 工作中
2: 工作中
3: 被中断，原因 shutdown


中断只会在进程处于等待事件时被投递。被中断进程恢复后，会在对应 `yield` 位置抛出 `simpy.Interrupt`。


## 中断后的清理

如果进程占用了资源，建议用 `with` 或 `try/finally` 确保释放。

```python
def job(env, resource):
    with resource.request() as req:
        yield req
        try:
            yield env.timeout(100)
        except simpy.Interrupt:
            print("任务被取消")
            raise
```

`with resource.request()` 会在离开上下文时释放资源。

## Environment.run

`env.run()` 有几种常见写法：

```python
env.run()                 # 运行到没有事件
env.run(until=10)         # 运行到仿真时间 10
env.run(until=event)      # 运行到某个事件被处理
```

当 `until` 是事件时，`run()` 会返回事件值。

```python
event = env.timeout(5, value="finished")
result = env.run(until=event)
print(result)
```

## step 和 peek

`step()` 可以只处理下一个事件，`peek()` 可以查看下一个事件的时间。

```python
while env.peek() < 100:
    env.step()
```

这两个接口适合：

- 调试事件队列。
- 和外部循环集成。
- 手动控制仿真推进。

普通模型优先使用 `run()`。

## 实时环境

默认 `Environment` 不等待真实时间。`RealtimeEnvironment` 可以让仿真时间和真实时间同步。

In [12]:
import simpy.rt


env = simpy.rt.RealtimeEnvironment(factor=1.0, strict=True)
event = env.timeout(2.0, value="timeout")
env.run(until=event)

'timeout'

含义：

- `factor=1.0`：1 个仿真时间单位对应 1 秒真实时间。
- `factor=0.1`：1 个仿真时间单位对应 0.1 秒真实时间。
- `strict=True`：如果事件处理慢到跟不上真实时间，会抛出错误。

实时环境适合演示、硬件交互、仿真可视化，不适合大量蒙特卡洛实验。

## 工具函数

### start_delayed

`simpy.util.start_delayed()` 可以延迟启动一个进程。

In [13]:
import simpy
from simpy.util import start_delayed


def task(env):
    print(f"{env.now}: task started")
    yield env.timeout(1)


env = simpy.Environment()
start_delayed(env, task(env), delay=5)
env.run()

5: task started


### subscribe_at

`simpy.util.subscribe_at()` 可以让当前进程在另一个事件发生时被中断，常用于订阅某个时间点或事件。

它不如 `yield event` 常用，但在需要“继续做当前事情，同时订阅外部事件”的模型中很有用。

## 调度顺序

当多个事件发生在同一个仿真时间，SimPy 会使用内部优先级和事件编号保证顺序稳定。通常不需要依赖这种细节，但在写测试时要注意：

- 同一时间点的输出顺序可能由事件创建顺序决定。
- 中断、初始化、普通事件可能有不同内部优先级。
- 如果业务逻辑依赖同一时刻的严格顺序，应显式建模优先级，而不是依赖偶然顺序。

## 常见错误

### 忘记 yield 资源请求

错误：

```python
req = resource.request()
yield env.timeout(5)
```

正确：

```python
with resource.request() as req:
    yield req
    yield env.timeout(5)
```

### 事件重复触发

一个普通事件只能成功或失败一次。

```python
event.succeed()
event.succeed()  # 错误
```

如果需要多次通知，应每次创建新事件，或用 `Store` 传递消息。

### 捕获中断后没有处理状态

```python
try:
    yield env.timeout(100)
except simpy.Interrupt:
    pass
```

这样虽然吞掉了中断，但可能留下业务状态不一致。更好的做法是记录取消原因、释放资源、更新指标，然后决定是否继续或重新抛出。

## 小结

SimPy 的事件机制可以理解为：

- `Event` 是等待条件。
- `Process` 是特殊事件，完成时触发。
- `Timeout` 是未来某个时间触发的事件。
- `Condition` 是组合事件。
- `Interrupt` 是向等待中的进程注入异常。
- `Environment` 是按时间顺序处理这些事件的调度器。